# Train the stroke-to-motif model on Google Colab

**Before running:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

What this notebook does:
1. checks that a GPU is attached
2. connects your Google Drive, so results survive if Colab disconnects
3. downloads the code from GitHub (the repo is public, no login needed)
4. gets `dataset.npz` from your Drive, or asks you to upload it once
5. trains, writing checkpoints and sample pictures straight into Drive
6. shows the loss curve and the latest samples

`dataset.npz` is not in git. Make it on your machine with `python generator/c_model/prepare.py human`.
Put it in Drive at `MyDrive/arachne/dataset.npz` once, and later runs pick it up automatically.

## 1. Settings
Change these, then run everything. `SCRIPT` is the training file inside `generator/c_model/` on that branch.

In [ ]:
BRANCH = "main"                 # or "gan-experiments"
SCRIPT = "train.py"             # the training script to run
ARGS = "--epochs 300"           # e.g. "--quick" for a 1-minute check, "--batch 8" if memory runs out
DRIVE_DIR = "arachne"           # folder in MyDrive for the dataset and results

## 2. Check the GPU
If this says `cuda available: False`, set the runtime type to GPU and run again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

## 3. Connect Google Drive
A pop-up asks for permission. Results go to `MyDrive/arachne/runs/<date-time>/`.

In [ ]:
from datetime import datetime
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive") / DRIVE_DIR
RUN = DRIVE / "runs" / f"{datetime.now():%Y-%m-%d_%H%M}_{BRANCH}_{Path(SCRIPT).stem}"
(RUN / "out").mkdir(parents=True, exist_ok=True)
(RUN / "checkpoints").mkdir(exist_ok=True)
print("results will be saved to", RUN)

## 4. Get the code

In [ ]:
!rm -rf /content/arachne-generator
!git clone --quiet --depth 1 --branch {BRANCH} https://github.com/nikolpapad/arachne-generator.git /content/arachne-generator
!pip install --quiet scikit-image
CODE = Path("/content/arachne-generator/generator/c_model")
!git -C /content/arachne-generator log --oneline -1

## 5. Get the dataset
Taken from Drive if it is there; otherwise upload `dataset.npz` from your computer (it is then also copied to Drive for next time).

In [ ]:
import shutil
saved = DRIVE / "dataset.npz"
if not saved.exists():
    from google.colab import files
    print("Choose generator/c_model/dataset.npz from your computer:")
    uploaded = files.upload()
    name = next(iter(uploaded))
    shutil.move(name, saved)
shutil.copy(saved, CODE / "dataset.npz")
shutil.copy(saved, RUN / "dataset.npz")  # keep a copy with the run, so you know what it was trained on

import numpy as np
d = np.load(CODE / "dataset.npz")
print({k: d[k].shape for k in d}, "| test figures:", int(d["test"].sum()))

## 6. Train
The training script writes into `out/` and `checkpoints/`; those are linked to the Drive run folder, so every checkpoint and sample sheet is saved as it is made.

Colab's free tier can disconnect after a while; if it does, the results so far are already in Drive.

In [ ]:
for name in ["out", "checkpoints"]:
    target = CODE / name
    if target.is_symlink() or target.is_file():
        target.unlink()
    elif target.exists():
        shutil.rmtree(target)
    target.symlink_to(RUN / name)

!cd {CODE} && python {SCRIPT} {ARGS} 2>&1 | tee {RUN}/train.out

## 7. Look at the results
Loss curves from every `*log.csv` the script wrote, and the most recent sample sheet (works for `train.py` and `train_gan.py`).

In [ ]:
import csv
import matplotlib.pyplot as plt
from IPython.display import Image, display

# each script names its files differently: train.py → train_log.csv / samples_*,
# train_gan.py → gan_log.csv / gan_samples_*
for log in sorted((RUN / "out").glob("*log.csv")):
    rows = list(csv.DictReader(open(log)))
    if not rows:
        continue
    epochs = [int(r["epoch"]) for r in rows]
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for key in rows[0]:
        if key != "epoch":
            ax.plot(epochs, [float(r[key]) for r in rows], label=key, marker=".")
    ax.set_title(log.name); ax.set_xlabel("epoch"); ax.legend(); ax.grid(alpha=.3)
    plt.show()

samples = sorted((RUN / "out").glob("*samples_*.png"), key=lambda f: f.stat().st_mtime)
if samples:
    print(samples[-1].name)
    display(Image(filename=str(samples[-1]), width=520))
else:
    print("no sample pictures yet in", RUN / "out")


## 8. Optional: download everything as one zip
Everything is already in Drive; this is only if you want it on your computer directly.

In [ ]:
archive = shutil.make_archive(f"/content/{RUN.name}", "zip", RUN)
from google.colab import files
files.download(archive)